# LC 322 — Coin Change
**Day-37 | Two Pointers + Review**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Build the answer bottom-up: the
minimum coins for amount `i` is 1 plus the minimum coins for
`i - coin`, over all valid coins. Small subproblems power larger
ones.
</div>

## Official Problem Statement

You are given an integer array `coins` representing coins of
different denominations and an integer `amount` representing a
total amount of money. Return the **fewest number of coins**
that you need to make up that amount. If that amount cannot be
made up by any combination of the coins, return `-1`. You may
assume that you have an **infinite number** of each kind of coin.

**Constraints:**
- `1 <= coins.length <= 12`
- `1 <= coins[i] <= 2^31 - 1`
- `0 <= amount <= 10^4`

## What This Is Actually Asking

Given unlimited coins of specific denominations, find the
minimum number of coins needed to make exact change for
`amount`. Greedy (always pick the largest coin) does not always
work — counterexample: coins=[1,3,4], amount=6: greedy gives
4+1+1=3 coins, but 3+3=2 coins is better. Dynamic programming
is required to guarantee the global optimum.

## Walk Through an Example by Hand

```
coins = [1, 5, 6, 9], amount = 11

dp[0] = 0   (0 coins to make 0)
dp[1] = dp[0]+1 = 1   (use coin 1)
dp[2] = dp[1]+1 = 2
dp[3] = 3
dp[4] = 4
dp[5] = min(dp[4]+1, dp[0]+1) = min(5,1) = 1   (use coin 5)
dp[6] = min(dp[5]+1, dp[1]+1, dp[0]+1) = min(2,2,1) = 1 (coin 6)
dp[7] = min(dp[6]+1, dp[2]+1, dp[1]+1) = min(2,3,2) = 2
dp[8] = 2
dp[9] = min(dp[8]+1, dp[4]+1, dp[3]+1, dp[0]+1) = 1 (coin 9)
dp[10]= min(dp[9]+1, dp[5]+1, dp[4]+1, dp[1]+1) = 2
dp[11]= min(dp[10]+1, dp[6]+1, dp[5]+1, dp[2]+1) = 2 (5+6)

Answer: dp[11] = 2
```

## The Picture

```
coins=[1,5,6,9]  amount=11

dp array (index = amount, value = min coins):

 amt:  0  1  2  3  4  5  6  7  8  9 10 11
  dp: [0, 1, 2, 3, 4, 1, 1, 2, 2, 1, 2, 2]
              ^
              dp[i] = min over all coins c where c <= i:
                         dp[i - c] + 1

Recurrence:
  dp[0] = 0
  dp[i] = min(dp[i - c] + 1)  for each coin c <= i
  dp[i] = inf if no coin can reach i

Think of it as: "what's the cheapest last coin I could use
  to land on amount i?"
```

## When To Use This Pattern

- When you need the **minimum/maximum count** to reach a target
  and choices can be reused, think **unbounded knapsack DP**.
- When greedy fails (optimal substructure but not greedy choice
  property), think **dynamic programming**.
- When the problem has overlapping subproblems (same sub-amounts
  computed repeatedly), think **bottom-up DP table**.
- When you see `amount <= 10^4` and `coins.length <= 12`,
  think **O(amount * len(coins)) is fast enough**.

## The Approach

Initialize a DP array of size `amount+1` with infinity, then
set `dp[0] = 0` (zero coins needed for zero amount). Iterate
amounts from 1 to `amount` inclusive. For each amount, try
every coin: if the coin fits (`coin <= i`), update `dp[i]` with
`min(dp[i], dp[i - coin] + 1)`. Return `dp[amount]` if it is
not infinity, else return `-1`.

In [ ]:
from typing import List
import math

In [ ]:
def test_harness(func):
    tests = [
        # (coins, amount, expected)
        ([1, 5, 6, 9],   11,   2),
        ([1, 2, 5],      11,   3),
        ([2],             3,  -1),
        ([1],             0,   0),
        ([1],             1,   1),
        ([186, 419, 83, 408], 6249, 20),  # edge: large amount
    ]
    passed = 0
    for coins, amount, expected in tests:
        result = func(coins, amount)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(f"{status} | coins={coins}, amount={amount} "
              f"| got={result}, expected={expected}")
    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def coin_change(coins: List[int], amount: int) -> int:
    """
    Bottom-up DP: build minimum coin counts from 0 to amount.

    Args:
        coins:  List of coin denominations (unlimited supply).
        amount: Target integer amount.

    Returns:
        Minimum number of coins to make amount, or -1 if
        impossible.

    Time:  O(amount * len(coins))  — double loop
    Space: O(amount)               — dp array
    """
    INF = float('inf')
    dp = [INF] * (amount + 1)
    dp[0] = 0

    # Debug: show setup
    print(f"[DEBUG] coins={coins}, amount={amount}")
    print(f"[DEBUG] dp[0..5]={dp[:6]}")

    pass

    # Debug: trace dp updates
    # for i in range(1, amount + 1):
    #     for coin in coins:
    #         if coin <= i and dp[i - coin] + 1 < dp[i]:
    #             dp[i] = dp[i - coin] + 1
    #             print(f"[DEBUG] dp[{i}]={dp[i]} via coin={coin}")

In [ ]:
# Uncomment and run when solution is ready
# test_harness(coin_change)

## Complexity

| Approach           | Time              | Space     | Notes                    |
|--------------------|-------------------|-----------|--------------------------|
| Brute force (DFS)  | O(S^n) worst      | O(n)      | Exponential, impractical |
| Memoized recursion | O(S * n)          | O(S)      | Top-down DP              |
| Bottom-up DP       | O(S * n)          | O(S)      | Optimal; iterative       |

*S = amount, n = number of coin denominations*

## Real World Connection

Coin change is the canonical example of the unbounded knapsack
problem, which appears in resource allocation at every tech
company. At AWS, packing Lambda functions into reserved
concurrency slots with minimum waste uses the same DP pattern.
In financial systems at Citi, optimal lot-sizing for trades
(minimize the number of order fills to reach a target notional)
mirrors coin change exactly. Any DE pipeline that minimizes the
number of batch runs or partitions to cover all data fits this
mold.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra